In [ ]:
import os

print(os.getcwd())

In [ ]:
import numpy as np
import astropy.units as u
from astropy.constants import G
import matplotlib.pyplot as plt

# Datos

In [ ]:
today_Myr = 13700

Table A3 from Mollá et al. (2015)
https://ui.adsabs.harvard.edu/abs/2015MNRAS.451.3693M

In [ ]:
MW = {}
MW['R']     = np.array([         0,      1,     2,     3,     4,     5,     6,    7,    8,    9,   10,   11,   12,   13,   14,    15,   16,     17,     18,     19,     20])
MW['stars'] = 10**np.array([np.NaN, np.NaN, np.NaN, 2.43,  2.50,  2.40,  2.25, 2.09, 1.95, 1.79, 1.69, 1.51, 1.38, 1.25, 1.09,  0.94, 0.80, np.NaN, np.NaN, np.NaN, np.NaN])
MW['HI']    =     np.array([  9.41,   3.97,   2.37, 2.39,  3.86,  5.06,  5.04, 5.44, 5.69, 7.69, 6.52, 6.16, 5.63, 4.83, 3.65,  2.96, 2.42,   2.15,   1.61,   1.18,    1.1])
MW['H2']    =     np.array([  0.30,   3.82,   5.18, 3.48,  5.69,  8.28,  8.47, 4.59, 3.15, 2.44, 1.96, 1.24, 0.99, 0.57, 0.82,  1.09,  .20,    .13,    .08,    .03, np.NaN])
MW['gas']   = MW['HI'] + MW['H2']
MW['SFR']   = 10**np.array([  -.37,   .603,   .706, .983, 1.163, 1.185, 1.181, .963, .723, .594,  .51, .403, .006, .183, -.26, -.132, -.52,   -.68,   -.89,  -1.37,  -1.37])
MW['OH']    =     np.array([  9.02,   8.86,   8.74, 8.62,  8.82,  8.83,  8.77, 8.69, 8.56, 8.60, 8.45, 8.41, 8.44, 8.44, 8.42,  8.14, 8.14,   8.19,   7.96, np.NaN, np.NaN])
MW['total'] = MW['gas'] + MW['stars']

# Modelo de capas cilindricas


## Verificación intermedia

In [ ]:
# Parámetros de prueba
Area = np.pi * (8000 * u.pc) ** 2
Mg_test = 2e9 * u.Msun
Ms_test = 5e8 * u.Msun
sigmas = np.linspace(5, 50, 100) * u.km / u.s


def verify_physics(Mg, Ms, sigma):
    # Gravedad (pc/Myr^2)
    surf_density = (Mg + Ms) / Area
    g = (2 * np.pi * G * surf_density).to(u.pc / u.Myr**2)

    # Estructura (z0 en pc, rho0 en Msun/pc^3)
    z0 = (sigma**2 / g).to(u.pc)
    rho0 = (Mg / (2 * Area * z0)).to(u.Msun / u.pc**3)

    # 3. Energía (erg)
    E_total = (2.5 * Mg * sigma**2).to(u.erg)

    # Retornamos solo los valores numéricos en las unidades deseadas
    return z0.value, rho0.value, E_total.value


# Ejecutar verificación
results = [verify_physics(Mg_test, Ms_test, s) for s in sigmas]
z0_vals, rho0_vals, E_vals = zip(*results)

# Gráficas de verificación
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(sigmas.value, z0_vals)
ax[0].set_title("Expansión del disco: $z_0$ vs $\sigma$")
ax[0].set_ylabel("z0 [pc]")
ax[0].set_xlabel("sigma [km/s]")

ax[1].plot(sigmas.value, rho0_vals)
ax[1].set_title("Densidad central: $\\rho_0$ vs $\sigma$")
ax[1].set_ylabel("rho0 [$M_\odot/pc^3$]")
ax[1].set_xlabel("sigma [km/s]")
plt.tight_layout()
plt.show()

# Bloque principal

## Usando el $t_{cooling}$  según la fase de Sedov-Taylor y SFE constante

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import astropy.units as u
from astropy.constants import G
import matplotlib.pyplot as plt


class GalacticEvolution_Refined:
    def __init__(self):
        # Configuración de unidades (Msun, pc, Myr)
        self.G_sim = G.to(u.pc**3 / (u.Msun * u.Myr**2)).value
        self.Area = np.pi * (8000) ** 2

        # Constantes físicas (Ecuación 2.20 Tesis Mario)
        self.mH_sim = (1.67e-27 * u.kg).to(u.Msun).value
        self.kB_sim = (
            (1.38e-23 * u.J / u.K).to(u.Msun * u.pc**2 / (u.Myr**2 * u.K)).value
        )

        # Parámetros del artículo
        self.a_idx = -0.9  # Índice 'a' de la curva de enfriamiento
        self.Ta = 1e5  # Temperatura de referencia (K)
        self.Lambda_a = 1e-22  # erg * cm^3 / s
        self.n0 = 1.0  # Densidad numérica ambiental (cm^-3)

        # Factores de conversión para el cálculo de t_cool (CGS)
        self.mH_cgs = 1.67e-24  # g
        self.kB_cgs = 1.38e-16  # erg/K
        self.n0_cgs = self.n0  # cm^-3
        self.Lambda_a_cgs = self.Lambda_a  # erg * cm^3 / s

        # E0 (Energía de supernova)
        self.E0_cgs = 1e51  # erg

        v0_phys = 220 * u.km / u.s
        self.v0_sim_sq = v0_phys.to(u.pc / u.Myr).value ** 2
        self.E_sn_sim = (1e51 * u.erg).to(u.Msun * u.pc**2 / u.Myr**2).value

        # Parámetros de estabilidad
        self.eps = 1e-4
        self.M_floor = 1e6

        self.eta_sn = 0.01
        self.sfe = 0.02  # SFE constante
        self.infall_rate = 1.0e6
        self.tau_inf = 1500.0  # Tau para un infall exponencial

        self.chi = 0.1  # Parámetro chi de metalicidad/eficiencia

    def get_physics(self, Mg, Ms, E):
        Mg_eff = Mg + self.eps
        E_eff = E + self.eps

        sigma_sq = 0.4 * (E_eff / Mg_eff)

        g = 2 * np.pi * self.G_sim * (Mg_eff + Ms) / self.Area
        z0 = sigma_sq / (g + 1e-15)

        rho0 = Mg_eff / (2 * self.Area * z0 + 1e-15)
        t_ff = np.sqrt((3 * np.pi) / (32 * self.G_sim * (rho0 + 1e-15)))

        # Cálculo de T_COOL (Ecuación 2.20)
        # t_c^[(11-6a)/5] = [81(1-a) m_H / (1600 * n_0 lambda_a )] * [9 m_H / (80 * k_B T_a)]^-a * [16 * chi * E_0 / (375 * n_0* m_H)]^[2(1-a)/5]

        a = self.a_idx

        # Término constante 1
        term_const = (81 * (1 - a) * self.mH_cgs) / (
            1600 * self.n0_cgs * self.Lambda_a_cgs
        )

        # Término de temperatura (Término 2)
        term_temp = ((9 * self.mH_cgs) / (80 * self.kB_cgs * self.Ta)) ** (-a)

        # Término de energía E0 (Término 3) - Usamos E0 constante 1e51 erg y n0 sin mu
        term_E = (16 * self.chi * self.E0_cgs) / (375 * self.n0_cgs * self.mH_cgs)

        expo_E = (2 / 5) * (1 - a)
        expo_final = (1 / 5) * (11 - 6 * a)

        # Despejamos t_cool en segundos y convertimos a Myr
        t_cool_s = (term_const * term_temp * (term_E**expo_E)) ** (1 / expo_final)
        t_cool = t_cool_s / 3.154e13  # de s a Myr

        # Aseguramos un valor físico razonable (en Myr)
        t_cool = np.clip(t_cool, 0.1, 13000.0)

        return sigma_sq, t_ff, z0, t_cool

    def derivatives(self, t, y):
        Mg, Ms, E = y
        sigma_sq, t_ff, z0, t_cool = self.get_physics(Mg, Ms, E)

        # infall = self.infall_rate * np.exp(-t / self.tau_inf) # Para un infall exponencial
        infall = self.infall_rate  # Infall constante
        sfr = self.sfe * max(Mg, 0) / t_ff

        dMg = infall - sfr
        dMs = sfr

        # E_DOT
        e_dot_in = infall * (0.5 * self.v0_sim_sq + sigma_sq)
        e_dot_sn = sfr * self.eta_sn * self.E_sn_sim
        e_dot_sfr = sfr * (E / (Mg + self.eps))
        e_dot_cool = E / t_cool

        dE = e_dot_in + e_dot_sn - e_dot_sfr - e_dot_cool

        return [dMg, dMs, dE]


# Ejecución
model = GalacticEvolution_Refined()
sigma_init_sim = (15 * u.km / u.s).to_value(u.pc / u.Myr)
E_init = 2.5 * 2e9 * (sigma_init_sim**2)
y0 = [2e9, 5e8, E_init]

sol = solve_ivp(model.derivatives, [0, today_Myr], y0, method="LSODA", rtol=1e-6)

# Procesamiento y gráficas
Mg_s, Ms_s, E_s = sol.y
sigma_res = np.sqrt(0.4 * np.maximum(E_s, 0) / np.maximum(Mg_s, 1)) * 0.9778

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(sol.t, Mg_s / 1e9, label="Gas")
ax[0].plot(sol.t, Ms_s / 1e9, label="Estrellas")
ax[0].set_title("Evolución de Masas")
ax[0].set_xlabel("Tiempo (Myr)")
#ax[0].set_xscale("log")
ax[0].set_ylabel("$10^9 M_\odot$")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

ax[1].plot(sol.t, sigma_res, color="green")
ax[1].set_title(r"Dispersión de velocidades $\sigma$ (km/s)")
ax[1].set_xlabel("Tiempo (Myr)")
ax[1].set_xscale("log")
ax[1].set_ylabel("$\sigma$ (km/s)")
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## CÁLCULOS DE DIAGNÓSTICO

In [ ]:
# Recuperamos las masas y energía de la solución
Mg_v, Ms_v, E_v = sol.y

# Frecuencia epicíclica (kappa)
# Suponiendo una curva de rotación plana V_c = 220 km/s a R = 8000 pc
v_c_sim = (220 * u.km / u.s).to_value(u.pc / u.Myr)
kappa = np.sqrt(2) * v_c_sim / 8000.0  # Frecuencia para un potencial logarítmico

# Parámetro de estabilidad de Toomre Q
# Q = (sigma * kappa) / (pi * G * Sigma_gas)
# Sigma_gas es la densidad superficial de masa (Mg / Area)
sigma_v = np.sqrt(0.4 * np.maximum(E_v, 0) / np.maximum(Mg_v, 1))
Sigma_gas = Mg_v / model.Area
Q_toomre = (sigma_v * kappa) / (np.pi * model.G_sim * Sigma_gas)

# Ratio Virial (K / |U|)
# K = 1.5 * Mg * sigma^2 y U = Mg * g * z0
# Por construcción (E = 2.5 Mg s^2), este ratio debería ser estable cerca de 1.5
g_v = 2 * np.pi * model.G_sim * (Mg_v + Ms_v) / model.Area
z0_v = sigma_v**2 / g_v
K_kin = 1.5 * Mg_v * sigma_v**2
U_pot = Mg_v * g_v * z0_v
virial_ratio = K_kin / (U_pot + 1e-10)

# Estabilidad geométrica (z0 / R)
# Mide qué tan "delgado" es el disco comparado con su radio (8 kpc)
thickness_ratio = z0_v / 8000.0

# PLOTS DE VERIFICACIÓN
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot Q de Toomre
axes[0].plot(sol.t, Q_toomre, color="purple", lw=2)
axes[0].axhline(1, color="red", linestyle="--", label="Límite Inestabilidad")
axes[0].set_title("Parámetro $Q$ de Toomre")
axes[0].set_ylabel("Q")
axes[0].set_xlabel("Tiempo (Myr)")
axes[0].set_xscale("log")
axes[0].legend()

# Plot Ratio Virial
axes[1].plot(sol.t, virial_ratio, color="orange", lw=2)
axes[1].set_title("Ratio Virial ($K/U$)")
axes[1].set_ylabel("Ratio")
axes[1].set_xlabel("Tiempo (Myr)")

# Plot Espesor del Disco
axes[2].plot(sol.t, thickness_ratio, color="brown", lw=2)
axes[2].axhline(0.1, color="black", linestyle=":", label="Disco Delgado (10%)")
axes[2].set_title("Espesor Relativo ($z_0/R$)")
axes[2].set_ylabel("Ratio $z_0/R$")
axes[2].set_yscale("log")
axes[2].set_xlabel("Tiempo (Myr)")
axes[2].set_xscale("log")
axes[2].legend()

plt.tight_layout()
plt.show()

# Infall exponencial

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import astropy.units as u
from astropy.constants import G
import matplotlib.pyplot as plt


class GalacticEvolution_Refined:
    def __init__(self, radius_pc=8000, tau_inf=12000.0):
        self.G_sim = G.to(u.pc**3 / (u.Msun * u.Myr**2)).value

        # GEOMETRÍA: Anillo local de 1kpc de ancho
        self.dr = 1000
        self.Area = 2 * np.pi * radius_pc * self.dr

        # INFALL RADIAL (Perfil de densidad superficial)
        self.t_univ = 13800.0
        Rd = 3000  # Radio de escala del disco (típico MW)

        sigma_0_inf = 2000.0
        sigma_r_total = sigma_0_inf * np.exp(-radius_pc / Rd)

        self.M_total_anillo = sigma_r_total * self.Area

        self.tau_inf = tau_inf
        self.I0 = self.M_total_anillo / self.tau_inf

        # Parámetros Físicos
        self.v0_sim_sq = (220 * u.km / u.s).to(u.pc / u.Myr).value ** 2
        self.E_sn_sim = (1e51 * u.erg).to(u.Msun * u.pc**2 / u.Myr**2).value
        self.sfe = 0.004
        self.eta_sn = 0.01
        self.eps = 1e-4

        # Constantes para T_COOL
        self.a_idx = -0.9  # Ajustado al valor del artículo
        self.chi = 0.1
        self.Ta = 1e5  # K
        self.Lambda_a_cgs = 1e-22
        self.n0_cgs = 1.0
        self.mH_cgs = 1.67e-24
        self.kB_cgs = 1.38e-16
        self.E0_cgs = 1e51  # Energía SN constante

    def get_physics(self, Mg, Ms, E):
        Mg_eff, E_eff = Mg + self.eps, E + self.eps
        sigma_sq = 0.4 * (E_eff / Mg_eff)
        g = 2 * np.pi * self.G_sim * (Mg_eff + Ms) / self.Area
        z0 = sigma_sq / (g + 1e-15)  # Usamos g directamente para evitar singularidades
        rho0 = Mg_eff / (2 * self.Area * z0 + 1e-15)
        t_ff = np.sqrt((3 * np.pi) / (32 * self.G_sim * (rho0 + 1e-15)))

        # Cálculo de T_COOL (Ecuación 2.20)
        # t_c^[(11-6a)/5] = [81(1-a) m_H / (1600 * n_0 lambda_a )] * [9 m_H / (80 * k_B T_a)]^-a * [16 * chi * E_0 / (375 * n_0 * m_H)]^[2(1-a)/5]
        a = self.a_idx

        # Término constante 1
        term_const = (81 * (1 - a) * self.mH_cgs) / (
            1600 * self.n0_cgs * self.Lambda_a_cgs
        )

        # Término de temperatura (Término 2)
        term_temp = ((9 * self.mH_cgs) / (80 * self.kB_cgs * self.Ta)) ** (-a)

        # Término de energía E0 (Término 3)
        term_E = (16 * self.chi * self.E0_cgs) / (375 * self.n0_cgs * self.mH_cgs)

        expo_E = (2 / 5) * (1 - a)
        expo_final = (1 / 5) * (11 - 6 * a)

        # Despejamos t_cool en segundos y convertimos a Myr
        t_cool_s = (term_const * term_temp * (term_E**expo_E)) ** (1 / expo_final)
        t_cool = t_cool_s / 3.154e13

        # Clip de seguridad para estabilidad del integrador
        t_cool = np.clip(t_cool, 0.1, 13000.0)

        return sigma_sq, t_ff, z0, t_cool

    def derivatives(self, t, y):
        Mg, Ms, E = y
        sigma_sq, t_ff, z0, t_cool = self.get_physics(Mg, Ms, E)

        # Infall Exponencial
        infall = self.I0 * np.exp(-t / self.tau_inf)
        sfr = self.sfe * max(Mg, 0) / t_ff

        dMg = infall - sfr
        dMs = sfr
        e_dot_in = infall * (0.5 * self.v0_sim_sq + sigma_sq)
        e_dot_sn = sfr * self.eta_sn * self.E_sn_sim
        e_dot_sfr = sfr * (E / (Mg + self.eps))
        e_dot_cool = E / t_cool
        dE = e_dot_in + e_dot_sn - e_dot_sfr - e_dot_cool
        return [dMg, dMs, dE]


# Ejecución (8 kpc)
model_8kpc = GalacticEvolution_Refined(radius_pc=8000)
sigma_init = (15 * u.km / u.s).to_value(u.pc / u.Myr)
y0 = [0, 0, 2.5 * 2e9 * (sigma_init**2)]

sol = solve_ivp(model_8kpc.derivatives, [0, today_Myr], y0, method="LSODA", rtol=1e-6)

t_v = sol.t
Mg_v = sol.y[0]
Ms_v = sol.y[1]
E_v = sol.y[2]
sigma_v = np.sqrt(0.4 * E_v / (Mg_v + 1e-6))

## Cálculos para verificación de determinados valores ( lo ideal es calcularlos a mano y ver si coinciden)

In [ ]:
# Valores de prueba
M_g_test, M_s_test, E_test = 2e9, 5e8, 1e12  # Valores arbitrarios para el test

# Usamos el modelo de 8kpc definido arriba
s_sq, tff, z_zero, t_c = model_8kpc.get_physics(M_g_test, M_s_test, E_test)

print("--- VERIFICACIÓN DE FÓRMULAS ---")
print(f"Masa Gas: {M_g_test:.2e} Msun")
print(f"Masa Estrellas: {M_s_test:.2e} Msun")
print(f"Sigma_gas: {M_g_test / model_8kpc.Area:.4e} Msun/pc^2")
print(f"z0 (Altura escala): {z_zero:.2f} pc")
print(
    f"rho0 (Densidad central): {M_g_test / (2 * model_8kpc.Area * z_zero):.4e} Msun/pc^3"
)
print(f"t_ff (Caída libre): {tff:.2f} Myr")

# Cálculo para distintos radios

In [ ]:
radios_kpc = np.arange(2, 23, 2)
sigma_star = []
sigma_gas = []
sigma_sfr = []

print(f"{'Radio (kpc)':<12} | {'log10(Sigma_star)':<18} | {'Sigma_SFR':<18}")
print("-" * 55)

for r in radios_kpc:
    # Creamos un modelo para cada radio (cambia el Area y el Infall local si se desea)
    m = GalacticEvolution_Refined(radius_pc=r * 1000)

    # Suponemos que el Infall local decae radialmente como en un disco (opcional)
    # Por simplicidad aquí usamos el I0 global, pero Mollá usa Sigma_inf(R)
    res = solve_ivp(m.derivatives, [0, today_Myr], y0, method="RK45")

    # Valores finales (t = 5000 Myr)
    Mg_f, Ms_f, E_f = res.y[:, -1]
    _, t_ff_f, _, _ = m.get_physics(Mg_f, Ms_f, E_f)

    sfr_f = m.sfe * Mg_f / t_ff_f
    area_kpc2 = m.Area / 1e6  # convertir pc^2 a kpc^2

    # Densidades superficiales
    s_star = Ms_f / area_kpc2
    s_sfr = (sfr_f / 1e6) / area_kpc2  # Msun / yr / kpc^2

    l_s_star = np.log10(s_star + 1e-10)
    #l_s_sfr = np.log10(s_sfr + 1e-15)

    sigma_star.append(s_star)
    sigma_gas.append(Mg_f / area_kpc2)
    sigma_sfr.append(s_sfr)

    print(f"{r:<12} | {l_s_star:<18.4f} | {s_sfr:<18.4f}")

## Perfiles radiales

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(radios_kpc, sigma_star, "k-", label="model")
plt.plot(MW['R'], MW['stars']*1e6, "ko", label="Mollá et al. (2015)")
plt.xlabel("Radio (kpc)")
plt.ylabel("Densidad de estrellas $\Sigma_\star$ (M$_\odot$/yr/kpc$^2$)")
plt.yscale("log")
plt.title(f"Perfiles Radiales Finales (t={today_Myr/1000:g} Gyr)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(radios_kpc, sigma_gas, "k-", label="model")
plt.plot(MW['R'], MW['gas']*1e6, "ko", label="Mollá et al. (2015)")
plt.xlabel("Radio (kpc)")
plt.ylabel("Densidad de gas $\Sigma_{gas}$ (M$_\odot$/yr/kpc$^2$)")
plt.yscale("log")
plt.title(f"Perfiles Radiales Finales (t={today_Myr/1000:g} Gyr)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(radios_kpc, sigma_sfr, "k-", label="model")
plt.plot(MW['R'], MW['SFR']*1e6/1e9, "ko", label="Mollá et al. (2015)")
plt.xlabel("Radio (kpc)")
plt.ylabel("Formación estelar $\Sigma_{SFR}$ (M$_\odot$/yr/kpc$^2$)")
plt.yscale("log")
plt.title(f"Perfiles Radiales Finales (t={today_Myr/1000:g} Gyr)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(radios_kpc, np.array(sigma_gas)/np.array(sigma_sfr) *1e-9, "k-", label="model")
plt.plot(MW['R'], MW['gas'] / MW['SFR'], 'ko', "ko", label="Mollá et al. (2015)")
plt.xlabel("Radio (kpc)")
plt.ylabel("Tempo de consumo de gas (Gyr)")
plt.yscale('log')
plt.title(f"Perfiles Radiales Finales (t={today_Myr/1000:g} Gyr)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Masa inicial despreciable y $\tau \ = \ 7000$ 

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import astropy.units as u
from astropy.constants import G
import matplotlib.pyplot as plt


class GalacticEvolution_Refined:
    def __init__(self, radius_pc=8000, tau_inf=2000.0):
        self.G_sim = G.to(u.pc**3 / (u.Msun * u.Myr**2)).value
        self.radius_pc = radius_pc
        self.dr = 1000
        self.Area = 2 * np.pi * radius_pc * self.dr

        # Perfil de Infall Mollá
        Rd = 3500.0
        sigma_0_inf = 500.0
        sigma_r_total = sigma_0_inf * np.exp(-radius_pc / Rd)

        self.M_total_anillo = sigma_r_total * self.Area
        self.tau_inf = tau_inf
        self.I0 = self.M_total_anillo / self.tau_inf

        self.v0_sim_sq = (220 * u.km / u.s).to(u.pc / u.Myr).value ** 2
        self.E_sn_sim = (1e51 * u.erg).to(u.Msun * u.pc**2 / u.Myr**2).value
        self.sfe = 0.02
        self.eta_sn = 0.01
        self.eps = 1e-4

        # Constantes físicas para T_COOL (CGS)
        self.a_idx = -0.9  # Valor del artículo
        self.chi = 0.1
        self.Ta = 1e5  # K
        self.Lambda_a_cgs = 1e-22
        self.n0_cgs = 1.0
        self.mH_cgs = 1.67e-24
        self.kB_cgs = 1.38e-16
        self.E0_cgs = 1e51  # Energía SN constante

    def get_physics(self, Mg, Ms, E):
        Mg_eff = Mg + self.eps
        sigma_sq = 0.4 * (E / Mg_eff)

        g = 2 * np.pi * self.G_sim * (Mg_eff + Ms) / self.Area
        z0 = (sigma_sq / (g + 1e-15)) + 1.0  # El +1 evita divisiones por cero iniciales
        rho0 = Mg_eff / (2 * self.Area * z0)
        t_ff = np.sqrt((3 * np.pi) / (32 * self.G_sim * (rho0 + 1e-15)))

        a = self.a_idx

        # Término 1 (Constantes de enfriamiento)
        term1 = (81 * (1 - a) * self.mH_cgs) / (1600 * self.n0_cgs * self.Lambda_a_cgs)

        # Término 2 (Temperatura de referencia)
        term2 = ((9 * self.mH_cgs) / (80 * self.kB_cgs * self.Ta)) ** (-a)

        # Término 3 (Energía SN inyectada)
        term3 = (16 * self.chi * self.E0_cgs) / (375 * self.n0_cgs * self.mH_cgs)

        expo_E = (2 / 5) * (1 - a)
        expo_final = (1 / 5) * (11 - 6 * a)

        # Cálculo de t_cool en segundos y conversión a Myr
        t_cool_s = (term1 * term2 * (term3**expo_E)) ** (1 / expo_final)
        t_cool = t_cool_s / 3.154e13

        t_cool = np.clip(t_cool, 1.0, 1000.0)

        return sigma_sq, t_ff, z0, t_cool

    def derivatives(self, t, y):
        Mg, Ms, E = y
        sigma_sq, t_ff, z0, t_cool = self.get_physics(Mg, Ms, E)

        infall = self.I0 * np.exp(-t / self.tau_inf)
        sfr = self.sfe * max(Mg, 0) / t_ff

        dMg = infall - sfr
        dMs = sfr
        e_dot_in = infall * (0.5 * self.v0_sim_sq + sigma_sq)
        e_dot_sn = sfr * self.eta_sn * self.E_sn_sim
        e_dot_sfr = sfr * (E / (Mg + self.eps))
        e_dot_cool = E / t_cool

        return [dMg, dMs, e_dot_in + e_dot_sn - e_dot_sfr - e_dot_cool]


# Ejecución 8kpc
model_8kpc = GalacticEvolution_Refined(radius_pc=8000, tau_inf=7000.0)
sigma_init = (15 * u.km / u.s).to_value(u.pc / u.Myr)
y0 = [1e6, 0, 2.5 * 1e6 * (sigma_init**2)]
sol = solve_ivp(model_8kpc.derivatives, [0, today_Myr], y0, method="LSODA")

t_v = sol.t
Mg_v = sol.y[0]
Ms_v = sol.y[1]
E_v = sol.y[2]
sigma_v = np.sqrt(0.4 * E_v / (Mg_v + 1e-6))

## Verificación

In [ ]:
# Valores de prueba
M_g_test, M_s_test, E_test = 3e10, 0, 1e14  # Valores arbitrarios para el test

# Usamos el modelo de 8kpc definido arriba
s_sq, tff, z_zero, t_c = model_8kpc.get_physics(M_g_test, M_s_test, E_test)

print("--- VERIFICACIÓN DE FÓRMULAS ---")
print(f"Masa Gas: {M_g_test:.2e} Msun")
print(f"Masa Estrellas: {M_s_test:.2e} Msun")
print(f"Sigma_gas: {M_g_test / model_8kpc.Area:.4e} Msun/pc^2")
print(f"z0 (Altura escala): {z_zero:.2f} pc")
print(
    f"rho0 (Densidad central): {M_g_test / (2 * model_8kpc.Area * z_zero):.4e} Msun/pc^3"
)
print(f"t_ff (Caída libre): {tff:.2f} Myr")

## Cálculo de las densidades para distintos radios

In [ ]:
radios = np.arange(2, 22, 1)
log_sigma_star = []
sfr_vals = []

for r in radios:
    # Inside-out: tau aumenta con el radio
    tau_r = 1000 + 500 * r
    m = GalacticEvolution_Refined(radius_pc=r * 1000, tau_inf=tau_r)

    # Condiciones iniciales proporcionales al área del anillo
    Mg0_r = 1e4 * (r / 2)
    E0_r = (Mg0_r * sigma_init**2) / 0.4

    res = solve_ivp(m.derivatives, [0, today_Myr], [Mg0_r, 0, E0_r], method="LSODA")
    Mg_f, Ms_f, E_f = res.y[:, -1]
    _, t_ff_f, _, _ = m.get_physics(Mg_f, Ms_f, E_f)

    log_sigma_star.append(np.log10(Ms_f / m.Area + 1e-10))
    sfr_vals.append(m.sfe * Mg_f / t_ff_f)

# Normalización SFR respecto a 8kpc
idx_8 = np.argmin(np.abs(radios - 8))
log_psi_norm = np.log10(np.array(sfr_vals) / sfr_vals[idx_8])

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(radios, log_sigma_star, "o-")
ax[0].set_title(r"$\log \Sigma_{star}$")
ax[0].grid(True)
ax[0].axhline(0, color="red", ls=":")
ax[1].plot(radios, log_psi_norm, "s-", color="orange")
ax[1].set_title(r"$\log(\psi/\psi_\odot)$")
ax[1].grid(True)
plt.show()

In [ ]:
# Diagnóstico temporal a 8KPC
# Extracción de datos
t = sol.t
Mg, Ms, E = sol.y[0], sol.y[1], sol.y[2]

# Cálculos derivados
sigma_sq = 0.4 * E / (Mg + 1e-6)
sigma = np.sqrt(sigma_sq)
g = 2 * np.pi * model_8kpc.G_sim * (Mg + Ms) / model_8kpc.Area
z0 = sigma_sq / (g + 1e-15)

# Virial
K_turb = 1.5 * Mg * sigma_sq
U_grav = 1.0 * Mg * sigma_sq
virial_ratio = K_turb / (U_grav + 1e-15)

# Corrección de unidades a kappa
# Convertimos 220 km/s a pc/Myr de forma segura
v_circ_phys = 220 * (u.km / u.s)
v_circ_sim = v_circ_phys.to(u.pc / u.Myr).value

kappa = np.sqrt(2) * v_circ_sim / model_8kpc.radius_pc
Q = (sigma * kappa) / (np.pi * model_8kpc.G_sim * (Mg / model_8kpc.Area) + 1e-15)

#  Visualización
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Masas
axes[0, 0].plot(t, Mg, label="Gas", color="blue")
axes[0, 0].plot(t, Ms, label="Estrellas", color="orange")
axes[0, 0].set_title("Evolución de Masas")
axes[0, 0].legend()
axes[0, 0].grid(True)

# 2. Dispersión sigma
axes[0, 1].plot(t, sigma, color="green")
axes[0, 1].set_title(r"Dispersión $\sigma$ (pc/Myr)")
axes[0, 1].grid(True)

# 3. Factor Q
axes[0, 2].plot(t, Q, color="purple")
axes[0, 2].axhline(1.0, color="red", linestyle="--")
axes[0, 2].set_title("Q de Toomre")
axes[0, 2].set_ylim(0, 15)
axes[0, 2].grid(True)

# 4. Ratio Virial
axes[1, 0].plot(t, virial_ratio, color="darkgreen", lw=2)
axes[1, 0].axhline(1.5, color="black", linestyle="--", label="Estabilidad (1.5)")
axes[1, 0].set_title("Ratio Virial ($K/U$)")
axes[1, 0].set_ylim(0, 3)
axes[1, 0].grid(True)

# 5. Espesor Relativo
axes[1, 1].plot(t, z0 / model_8kpc.radius_pc, color="brown")
axes[1, 1].set_title("Espesor Relativo $z_0/R$")
axes[1, 1].set_ylim(0, 0.5)
axes[1, 1].grid(True)

# 6. log(Sigma_star)
axes[1, 2].plot(t, np.log10(Ms / model_8kpc.Area + 1e-10), color="black")
axes[1, 2].set_title(r"Evolución $\log \Sigma_{star}$")
axes[1, 2].grid(True)

plt.tight_layout()
plt.show()

# Variando $\tau$

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import astropy.units as u
from astropy.constants import G
import matplotlib.pyplot as plt


def ejecutar_diagnostico(modelo, t_span, y0):
    sol = solve_ivp(modelo.derivatives, t_span, y0, method="LSODA", rtol=1e-6)
    t = sol.t
    Mg, Ms, E = sol.y[0], sol.y[1], sol.y[2]

    # Cálculos derivados
    sigma_sq = 0.4 * E / (Mg + 1e-6)
    sigma = np.sqrt(sigma_sq)
    g = 2 * np.pi * modelo.G_sim * (Mg + Ms) / modelo.Area
    z0 = sigma_sq / (g + 1e-15)

    # Virial
    K_turb = 1.5 * Mg * sigma_sq
    U_grav = 1.0 * Mg * sigma_sq
    virial_ratio = K_turb / (U_grav + 1e-15)

    # Toomre Q
    v_circ_sim = (220 * u.km / u.s).to(u.pc / u.Myr).value
    kappa = np.sqrt(2) * v_circ_sim / modelo.radius_pc
    Q = (sigma * kappa) / (np.pi * modelo.G_sim * (Mg / modelo.Area) + 1e-15)

    return t, Mg, Ms, sigma, Q, virial_ratio, z0


# Configuración
t_span = [0, today_Myr]
radius_8kpc = 8000
sigma_init_val = (15 * u.km / u.s).to_value(u.pc / u.Myr)
# Masa inicial despreciable
y0_despreciable = [1e4, 0, (1e4 * sigma_init_val**2) / 0.4]

# Instanciamos los dos casos
model_fast = GalacticEvolution_Refined(
    radius_pc=radius_8kpc, tau_inf=1000.0
)  # Tau pequeño
model_const = GalacticEvolution_Refined(
    radius_pc=radius_8kpc, tau_inf=1e9
)  # tau -> infinito

# Ejecutamos ambos casos (usando la función de diagnóstico definida previamente)
data_fast = ejecutar_diagnostico(model_fast, t_span, y0_despreciable)
data_const = ejecutar_diagnostico(model_const, t_span, y0_despreciable)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

labels = [r"Infall Rápido ($\tau=1000$)", "Infall Constante"]
datasets = [data_fast, data_const]
colors_gas = ["blue", "cyan"]
colors_star = ["orange", "red"]
styles = ["-", "--"]

for i, data in enumerate(datasets):
    t, Mg, Ms, sigma, Q, virial, z0 = data

    # Masas (Escala logarítmica para ver el caso constante)
    axes[0, 0].plot(t, Mg, color=colors_gas[i], ls=styles[i], label=f"Gas {labels[i]}")
    axes[0, 0].plot(
        t, Ms, color=colors_star[i], ls=styles[i], label=f"Estrellas {labels[i]}"
    )
    axes[0, 0].set_yscale("log")
    axes[0, 0].set_title("Evolución de Masas")
    axes[0, 0].set_xlabel("Tiempo (Myr)")
    axes[0, 0].set_ylabel("Masa ($M_\odot$)")

    # Dispersión sigma
    axes[0, 1].plot(t, sigma, ls=styles[i], label=labels[i])
    axes[0, 1].set_title(r"Dispersión $\sigma$")
    axes[0, 1].set_xlabel("Tiempo (Myr)")
    axes[0, 1].set_ylabel("$\sigma$ (pc/Myr)")

    # Factor Q de Toomre
    axes[0, 2].plot(t, Q, ls=styles[i], label=labels[i])
    axes[0, 2].axhline(1.0, color="red", ls=":", alpha=0.5)
    axes[0, 2].set_title("Parámetro Q de Toomre")
    axes[0, 2].set_xlabel("Tiempo (Myr)")
    axes[0, 2].set_ylabel("Q (adimensional)")
    axes[0, 2].set_yscale("log")  # Q puede variar órdenes de magnitud

    # Ratio Virial
    axes[1, 0].plot(t, virial, ls=styles[i], label=labels[i])
    axes[1, 0].axhline(1.5, color="black", ls=":", alpha=0.7)
    axes[1, 0].set_title("Ratio Virial ($K/U$)")
    axes[1, 0].set_xlabel("Tiempo (Myr)")
    axes[1, 0].set_ylabel("Ratio")
    axes[1, 0].set_ylim(0, 3)

    # Espesor Relativo
    axes[1, 1].plot(t, z0 / radius_8kpc, ls=styles[i], label=labels[i])
    axes[1, 1].set_title("Espesor Relativo del Disco")
    axes[1, 1].set_xlabel("Tiempo (Myr)")
    axes[1, 1].set_ylabel("$z_0 / R$")

    # log(Sigma_star)
    area_sim = model_fast.Area
    axes[1, 2].plot(t, np.log10(Ms / area_sim + 1e-10), ls=styles[i], label=labels[i])
    axes[1, 2].set_title(r"Densidad Superficial Estelar $\log \Sigma_{\star}$")
    axes[1, 2].set_xlabel("Tiempo (Myr)")
    axes[1, 2].set_ylabel(r"$\log(M_\odot / pc^2)$")

# Ajustes finales de estilo
for ax in axes.flat:
    ax.legend(fontsize="x-small")
    ax.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

# En construcción

# SFR variable

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.special import erfc  # Función de error complementaria
import astropy.units as u
from astropy.constants import G, k_B, m_p
import matplotlib.pyplot as plt


class GalacticEvolution_Turbulent:
    def __init__(self):
        # UNIDADES Y CONSTANTES
        self.G_sim = G.to(u.pc**3 / (u.Msun * u.Myr**2)).value
        self.Area = np.pi * (8000) ** 2

        v0_phys = 220 * u.km / u.s
        self.v0_sim_sq = v0_phys.to(u.pc / u.Myr).value ** 2
        self.E_sn_sim = (1e51 * u.erg).to(u.Msun * u.pc**2 / u.Myr**2).value

        # Parámetros térmicos para la velocidad del sonido (c_s)
        # Asumimos gas a T ~ 10^4 K (Warm Neutral Medium). Es una buena aproximación??
        T_eff = 10000 * u.K
        self.cs_sq = (k_B * T_eff / (0.6 * m_p)).to(u.pc**2 / u.Myr**2).value

        self.eps = 1e-4
        self.eta_sn = 0.01
        self.infall_rate = 1.0e6

        # Parámetros Enfriamiento
        self.a_idx = 0.5
        self.chi = 0.1

    def get_physics(self, Mg, Ms, E):
        Mg_eff = Mg + self.eps
        E_eff = E + self.eps

        #  DISPERSIÓN Y GEOMETRÍA
        sigma_sq = 0.4 * (E_eff / Mg_eff)
        g = 2 * np.pi * self.G_sim * (Mg_eff + Ms) / self.Area
        z0 = (sigma_sq / g) + 1.0

        rho0 = Mg_eff / (2 * self.Area * z0)
        t_ff = np.sqrt((3 * np.pi) / (32 * self.G_sim * (rho0 + 1e-15)))

        #  SFR VARIABLE (LOG-NORMAL)
        # Número de Mach (sigma / c_s)
        mach_sq = max(sigma_sq / self.cs_sq, 0.01)

        # Parámetro Virial: Competición gravedad vs dispersión
        # alpha = sigma^2 / (g * z0). En equilibrio hidrostático puro sería ~1.
        alpha_vir = sigma_sq / (g * z0 + 1e-10)

        # Varianza de la log-normal (s^2) y densidad crítica (s_crit)
        # b = 0.4 es un valor estándar para forzamiento turbulento mixto
        s2 = np.log(1 + (0.4**2 * mach_sq))
        s_crit = np.log(max(alpha_vir * mach_sq, 1e-4))

        # SFE dinámica usando erfc
        sfe_dyn = 0.5 * erfc(
            (s_crit - s2 / 2) / np.sqrt(2 * s2 + 1e-15)
        )  # erfc = 1 - erf
        sfe_dyn = np.clip(sfe_dyn, 1e-6, 0.2)  # Evitamos que sea 0 o absurdamente alta

        # COOLING ANALÍTICO
        term_const = (81 * (1 - self.a_idx)) / 1600
        term_E = (16 * self.chi * (E_eff / Mg_eff)) / 375
        t_cool = (term_const * (term_E ** (0.4 * (1 - self.a_idx)))) ** (
            1 / (0.2 * (11 - 6 * self.a_idx))
        )
        t_cool = np.clip(t_cool, 1.0, 1000.0)

        return sigma_sq, t_ff, z0, t_cool, sfe_dyn

    def derivatives(self, t, y):
        Mg, Ms, E = y
        sigma_sq, t_ff, z0, t_cool, sfe_dyn = self.get_physics(Mg, Ms, E)

        # SFR ahora usa la sfe_dyn calculada
        sfr = sfe_dyn * max(Mg, 0) / t_ff

        dMg = self.infall_rate - sfr
        dMs = sfr

        e_dot_in = self.infall_rate * (0.5 * self.v0_sim_sq + sigma_sq)
        e_dot_sn = sfr * self.eta_sn * self.E_sn_sim
        e_dot_sfr = sfr * (E / (Mg + self.eps))
        e_dot_cool = E / t_cool

        dE = e_dot_in + e_dot_sn - e_dot_sfr - e_dot_cool

        return [dMg, dMs, dE]


#  EJECUCIÓN Y CÁLCULO DE HISTORIA
model = GalacticEvolution_Turbulent()
sigma_init_sim = (15 * u.km / u.s).to_value(u.pc / u.Myr)
y0 = [2e9, 5e8, 2.5 * 2e9 * (sigma_init_sim**2)]

sol = solve_ivp(model.derivatives, [0, today_Myr], y0, method="RK45", rtol=1e-5)

#  EXTRACCIÓN DE DATOS PARA GRÁFICAS ADICIONALES
sfr_val = []
sfe_val = []
for i in range(len(sol.t)):
    _, t_ff, _, _, sfe_dyn = model.get_physics(sol.y[0, i], sol.y[1, i], sol.y[2, i])
    sfr_val.append(sfe_dyn * sol.y[0, i] / t_ff)
    sfe_val.append(sfe_dyn)

#  GRÁFICAS
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Masas
axes[0, 0].plot(sol.t, sol.y[0] / 1e9, label="Gas")
axes[0, 0].plot(sol.t, sol.y[1] / 1e9, label="Estrellas")
axes[0, 0].set_title("Evolución de Masas")
axes[0, 0].legend()

# Sigma
sigma_km_s = np.sqrt(0.4 * np.maximum(sol.y[2], 0) / np.maximum(sol.y[0], 1)) / 1.0227
axes[0, 1].plot(sol.t, sigma_km_s, color="green")
axes[0, 1].set_title("Dispersión $\sigma$ (km/s)")

# SFR
axes[1, 0].plot(sol.t, sfr_val, color="blue")
axes[1, 0].set_title("SFR ($M_\odot/Myr$)")

# SFE
axes[1, 1].plot(sol.t, sfe_val, color="red")
axes[1, 1].set_title("SFE Dinámica (Log-normal)")

plt.tight_layout()
plt.savefig("mi_galaxia_SFR_variable.png", dpi=300, bbox_inches="tight")